# An Interpreter for a Simple Programming Language

In this notebook we develop an interpreter for a small programming language.  We will utilize the parser generator
`Larkita` to define our language rules.  The parser generated by `Larkita` returns the abstract syntax tree of a
program as a nested tuple.  This notebook is a variant of `Interpreter.ipynb`, which uses `Lark`.

Compared with the minimal version of this language, the interpreter developed below supports the following
additional features, which are needed by the example programs `carmichael.sl`, `e.sl`, `factorial.sl`, `pi.sl`,
`power.sl`, `primes.sl`, `solve.sl`, and `sum-for.sl`:

1. **User defined functions** with parameters, local variables, recursion, and a `return` statement.
2. **`for` loops** of the shape `for (i := 1; i <= n; i := i + 1) { ... }`.
3. **Logical operators** `&&`, `||`, and `!`, so that a test can be an arbitrary Boolean expression.
4. **Floating point numbers** like `1.0`, `0.0`, and `1.0e-15`, plus integer literals with underscores like `10_000`.
5. **String literals**, so that `print` can be called as `print("1 + ... + ", n, " = ", s)`.
6. **Sets**: the empty set `{}`, enumerations `{1, 2, 3}`, ranges `{1 .. n}`, and set comprehensions
   such as `{ p * q : p in {2 .. n/2}, q in {p .. n/p} }`.  The operators `+` and `-` denote union and
   set difference when applied to sets.
7. **Built-in functions**: `print`, `read`, `len`, `arb`, `cos`, `sin`, `exp`, `sqrt`, `int`, and `float`.

In [ ]:
import math
import sys

from larkita import Larkita

Recursion in the interpreted language is implemented by recursion in `Python`.  Every function call of our
language consumes several `Python` stack frames, so we raise the recursion limit.  The recursive computation of
`solveRecursive` in `solve.sl` needs a nesting depth of about 90 calls of the interpreted language.

In [ ]:
sys.setrecursionlimit(100_000)

## Specification of the Grammar

Below is the grammar for our simple `C`-like language expressed as an *Extended Backus-Naur Form* (EBNF) string.

Our programming language allows both single-line (`//`) and multi-line (`/* */`) comments.  We instruct the parser
to entirely `%ignore` them alongside standard whitespace.

Note that we have to use a *raw string* here, because otherwise the escape sequence `\/` would not be allowed.

The grammar also determines the abstract syntax tree:  The string literals of an alternative form the *head* of the
tuple that is returned for this alternative, while all other symbols become its *children*.  A few remarks on the
individual parts of the grammar:

* The directive `%ignore: "(", ")", "{", "}", ";"` (note the colon) removes the parentheses, the braces, and the
  semicolon from the heads of *all* alternatives, since these literals carry no information.  Hence the alternative
  `"while" "(" expr ")" statement` yields the tuple `('while', test, stmnt)` and `"return" [expr] ";"` yields
  `('return',)` or `('return', expr)`.  An optional part that is missing is simply left out.
* Following the style of the original grammar, the alternatives of `statement` are written inline.  Only `block`,
  `assignment`, and `params` are rules of their own, and each of them for a reason: `block` is used by the
  function definition as well, `assignment` is used both by an assignment statement and by the two slots of a
  `for` loop, and `params` contains a repetition.
* An alternative that consists of a single symbol passes the abstract syntax tree of this symbol through.  Hence the
  alternative `assignment ";"` yields the tuple of the assignment, and a block `"{" program "}"` simply *is* its
  program, that is a tuple of the form `('program', stmnt, ...)`.
* An *alias* `-> name` replaces the head.  We use aliases where the head given by the literals would not be what we
  want:  Without its alias, the alternative `expr ";"` would return the expression itself, and the head of the
  `if`-`else` statement would be the tuple `('if', 'else')`.  Function calls, unary minus, and the different kinds
  of sets get aliases, too.
* The arguments of a function call are written inline as `[expr ("," expr)*]`.  The children of an optional part and
  of a repetition are spliced into the tuple of the alternative, so a function call yields
  `('call', ('IDENTIFIER', name), arg_1, ..., arg_n)`.
* A function definition is a *statement*.  Therefore a file may freely mix definitions and top level code,
  which is exactly what `carmichael.sl` does.
* The old non-terminal `bool_expr` has disappeared.  Since `carmichael.sl` contains the test
  `n == 1 || is_prime(n) == 1`, comparisons have to be *ordinary expressions* that can be combined with `&&`
  and `||`.  The precedence, from weakest to strongest binding, is now
  `||` $\prec$ `&&` $\prec$ `!` $\prec$ comparison $\prec$ `+`, `-` $\prec$ `*`, `/`, `%` $\prec$ unary `-`.
* Comparisons are *non-associative*: `a < b < c` is a syntax error, as it should be.
* The terminal `FLOAT` has to be listed before `NUMBER` is tried, and it demands at least one digit after the
  decimal point.  This is what keeps `{ 2 .. 10 }` from being read as the floating point number `2.` followed
  by garbage.
* The token `NUMBER` allows underscores as digit separators, because `carmichael.sl` and `primes.sl` write
  `10_000`.

In [ ]:
sl_grammar = r"""
program: statement*

statement: "function" IDENTIFIER "(" params ")" block
         | "if" "(" expr ")" statement
         | "if" "(" expr ")" statement "else" statement      -> "if-else"
         | "while" "(" expr ")" statement
         | "for" "(" assignment ";" expr ";" assignment ")" statement
         | "return" [expr] ";"
         | assignment ";"
         | expr ";"                                            -> "expr"
         | block

block:      "{" program "}"
assignment: IDENTIFIER ":=" expr
params:     (IDENTIFIER ("," IDENTIFIER)*)?

expr: disjunction

disjunction: disjunction "||" conjunction
           | conjunction

conjunction: conjunction "&&" negation
           | negation

negation: "!" negation
        | comparison

comparison: sum "==" sum
          | sum "!=" sum
          | sum "<=" sum
          | sum ">=" sum
          | sum "<"  sum
          | sum ">"  sum
          | sum

sum: sum "+" product
   | sum "-" product
   | product

product: product "*" factor
       | product "/" factor
       | product "%" factor
       | factor

factor: "-" factor -> neg
      | atom

atom: "(" expr ")"
    | NUMBER
    | FLOAT
    | STRING
    | IDENTIFIER
    | IDENTIFIER "(" [expr ("," expr)*] ")" -> "call"
    | set_expr

// Sets: the empty set, an enumeration, a range, and two forms of set comprehension.
set_expr: "{" "}"                              -> "set"
        | "{" expr ".." expr "}"
        | "{" expr ("," expr)* "}"             -> "set"
        | "{" expr ":" generators "}"          -> "comprehension"
        | "{" expr ":" generators "|" expr "}" -> "comprehension"

generators: generator ("," generator)*
generator:  IDENTIFIER "in" expr

// These literals are left out of the heads of all tuples.
%ignore: "(", ")", "{", "}", ";"

// Lexical definitions
%import common.CNAME -> IDENTIFIER
%import common.WS

FLOAT:  /[0-9](_?[0-9])*\.[0-9](_?[0-9])*([eE][+-]?[0-9]+)?/
NUMBER: /[0-9](_?[0-9])*/
STRING: /"(\\.|[^"\\])*"/

// In order to show how it's done, I have defined the tokens
// CPP_COMMENT and C_COMMENT describing comments myself.
CPP_COMMENT: /\/\/[^\n]*/
C_COMMENT:   "/*" /(.|\n)*?/ "*/"

// Ignore whitespace and comments
%ignore WS
%ignore C_COMMENT
%ignore CPP_COMMENT
"""

### The grammar is still `LR(1)`

It is tempting to believe that sets destroy the *Look-Ahead Left-to-Right* property, because the character `{`
now starts two completely different constructs:

```
{ f(x); }        // a block containing one statement
{ f(x) }         // a set containing one element
{ x := 1; }      // a block containing an assignment
```

But an `LR` parser does *not* have to decide anything when it sees the `{`.  It simply shifts the token, and
the resulting state keeps the items of `block` **and** of the four set rules alive at the same time.  A decision
is only forced when a *reduction* becomes due, and at that moment one token of context is enough:

| after reading | lookahead | reduction |
|:--------------|:----------|:----------|
| `{ f(x)`      | `;`       | the expression statement, so the `{` opened a block |
| `{ f(x)`      | `}`       | the enumeration of a set, so the `{` opened a set   |
| `{ f(x)`      | `,`       | the enumeration of a set with more elements to come |
| `{ x`         | `:=`      | `assignment`, so the `{` opened a block |
| `{ 1`         | `..`      | the range of a set                      |
| `{ n`         | `:`       | a set comprehension                     |

Exactly **one** input has no distinguishing token, namely the empty `{}`: there the parser must choose between
reducing the empty `program` (which would complete a `block`) and shifting the `}` (which completes the empty set).
`Larkita` reports this as a shift/reduce conflict and resolves it in favour of the *shift*, so `{}` denotes the
**empty set**.  That is precisely what our programs need, since `power.sl` writes `if (S == {})` and
`return { {} };`.  The price we pay is that an empty block `{ }` is now a syntax error; none of our programs
contains one.  As the canonical LR(1) automaton has a separate state for every context in which a statement can
start with `{`, this conflict shows up in several states.

The second shift/reduce conflict is the classical *dangling else*, and shifting again gives the usual rule:
an `else` binds to the innermost `if` that does not have one yet.

`Larkita` issues a `ConflictWarning` when it builds the parser below.

In [ ]:
parser = Larkita(sl_grammar, out_file='sl.out')

The file `sl.out` describes all states of the LR(1) automaton.  Its section `Conflicts` lists every conflict together
with the rules involved and the way it has been resolved.  The cell below prints this section.

In [ ]:
with open('sl.out') as handle:
    report = handle.read()
print(report[report.index('Conflicts\n'):report.index('States\n')])

## The Abstract Syntax Tree (AST)

The parser returns the abstract syntax tree as a nested tuple, so there is no need to convert a parse tree.

The following type aliases describe the data that our interpreter deals with.

* `Number` is the type of the numbers of our language: `Python` integers of arbitrary size and floating point numbers.
* `Value` is the type of everything a variable can be bound to.  In addition to numbers, this includes strings,
  truth values, and sets.  Sets are represented as `frozenset`s because our language has *sets of sets*
  (see `power.sl`) and only immutable objects can be members of a set.
* `NestedTuple` is the type of a node of the abstract syntax tree.  It is a tuple whose first component is the head.
  A token is represented as a pair consisting of the name of the token and its text, e.g. `('NUMBER', '42')`.

In [ ]:
Number      = int | float
Value       = int | float | bool | str | frozenset
NestedTuple = Value | tuple

The shapes of the tuples that are produced are the following:

| construct           | nested tuple                                                    |
|:--------------------|:----------------------------------------------------------------|
| program / block     | `('program', stmnt, ...)`                                       |
| function definition | `('function', ('IDENTIFIER', name), (',', param, ...), body)`, where every `param` has the form `('IDENTIFIER', name)` |
| assignment          | `(':=', ('IDENTIFIER', name), expr)`                            |
| expression statement | `('expr', expr)`                                               |
| return              | `('return', expr)`, or `('return',)` for a bare `return;`       |
| conditional         | `('if', test, stmnt)` resp. `('if-else', test, s1, s2)`         |
| loops               | `('while', test, body)`, `('for', init, test, update, body)`    |
| operators           | `(op, lhs, rhs)`, `('neg', arg)`, `('!', arg)`                   |
| function call       | `('call', ('IDENTIFIER', name), arg, ...)`                      |
| tokens              | `('NUMBER', text)`, `('FLOAT', text)`, `('STRING', text)`, `('IDENTIFIER', name)` |
| set                 | `('set', e, ...)`, `('..', lo, hi)`                             |
| comprehension       | `('comprehension', expr, generators)` or `('comprehension', expr, generators, condition)`, where `generators` has the form `(',', ('in', ('IDENTIFIER', name), set_expr), ...)` |

Note that the parameter list of a function and the generators of a comprehension have the head `','`, since this
is the only literal of their rules.  Let us look at an example.

In [ ]:
parser.parse('function f(n) { return n * f(n - 1); } S := { x * x : x in {1 .. 5} | x % 2 == 1 };')

The function `unescape` turns the raw text of a string literal into the string it denotes.  The text of a token of
type `STRING` still contains the enclosing double quotes.

* **Input:** `text` of type `str`, the contents of a string literal *without* the enclosing double quotes.
* **Output:** a `str` in which the escape sequences `\n`, `\t`, `\r`, `\"`, and `\\` have been replaced by the
  characters they stand for.

In [ ]:
def unescape(text: str) -> str:
    Escapes = { 'n': '\n', 't': '\t', 'r': '\r', '"': '"', '\\': '\\' }
    Result  = []
    i       = 0
    while i < len(text):
        if text[i] == '\\' and i + 1 < len(text):
            Result.append(Escapes.get(text[i+1], text[i+1]))
            i += 2
        else:
            Result.append(text[i])
            i += 1
    return ''.join(Result)

The external helper file `AST2Dot.ipynb` is used to represent our generated nested tuple ADT graphically.
If that file is not present, the cell below installs a small replacement so that the notebook stays runnable.

In [ ]:
try:
    get_ipython().run_line_magic('run', 'AST2Dot.ipynb')
except Exception as error:
    print(f'AST2Dot.ipynb could not be run ({error}), using the built-in fallback.')

if 'tuple2dot' not in globals():
    import graphviz

    def tuple2dot(ast: NestedTuple) -> 'graphviz.Digraph':
        dot     = graphviz.Digraph(node_attr={'shape': 'box', 'style': 'rounded'})
        counter = 0
        def add(node) -> str:
            nonlocal counter
            counter += 1
            name = f'n{counter}'
            if isinstance(node, tuple) and len(node) > 0:
                dot.node(name, label=str(node[0]))
                for child in node[1:]:
                    dot.edge(name, add(child))
            else:
                dot.node(name, label=repr(node))
            return name
        add(ast)
        return dot

The function `parse` takes a `file_name` as its sole argument.  The file is read, parsed into a nested tuple, and
visualized via `graphviz`.

* **Input:** `file_name` of type `str`, the name of a file containing a program of our language.
* **Output:** a `graphviz.Digraph` showing the abstract syntax tree.  As a side effect the source code and the
  nested tuple are printed.

In [ ]:
def parse(file_name: str):
    with open(file_name, 'r') as handle:
        program = handle.read()
    print(program)
    ast = parser.parse(program)
    print(ast)
    return tuple2dot(ast)

In [ ]:
parse('factorial.sl')

## The Runtime System

Executing a program of this language needs three pieces of state:

1. `Functions`, a global dictionary mapping a function name to the pair `(parameters, body)`.
2. `Values`, a dictionary mapping variable names to their values.  Every function call creates a *fresh* `Values`
   dictionary that contains only the parameters, so functions have local variables and cannot see the variables
   of their caller.
3. The `Python` call stack, which carries the recursion of the interpreted program.

A `return` statement can occur arbitrarily deep inside loops and blocks.  The cleanest way to jump out of all
of them at once is a `Python` exception that carries the result value.

In [ ]:
class ReturnException(Exception):
    """Raised by a `return` statement in order to leave the body of a function."""
    def __init__(self, value: Value | None):
        self.value = value

In [ ]:
Functions: dict[str, tuple[tuple[str, ...], NestedTuple]] = {}

### Printing values

The function `sort_key` provides an ordering that works for all values of our language, so that the elements of
a set are always printed in the same, predictable order.

* **Input:** `value` of type `Value`.
* **Output:** a `tuple` that can be compared with the key of any other value.  Numbers come first, then strings,
  then sets; sets are ordered by size and then by the sorted keys of their elements.

In [ ]:
def sort_key(value: Value) -> tuple:
    match value:
        case frozenset():
            return (2, len(value), tuple(sorted(sort_key(x) for x in value)))
        case str():
            return (1, value)
        case _:
            return (0, value)

The function `to_string` renders a value the way our language wants to see it.

* **Input:** `value` of type `Value`.
* **Output:** a `str`.  Sets are shown in the usual mathematical notation `{1, 2, 3}`, truth values are printed
  as `true` and `false`, and strings are printed without quotes.

In [ ]:
def to_string(value: Value) -> str:
    match value:
        case frozenset():
            Elements = sorted(value, key=sort_key)
            return '{' + ', '.join(to_string(x) for x in Elements) + '}'
        case bool():
            return 'true' if value else 'false'
        case str():
            return value
        case _:
            return str(value)

### Overloaded operators

Three of the arithmetical operators are *polymorphic* in our language and therefore get a function of their own.

`sl_add` implements the operator `+`.

* **Input:** `lhs` and `rhs` of type `Value`.
* **Output:** the union `lhs` $\cup$ `rhs` if both arguments are sets, the concatenation of the two arguments if
  one of them is a string, and the numerical sum otherwise.

`sl_sub` implements the operator `-`.

* **Input:** `lhs` and `rhs` of type `Value`.
* **Output:** the set difference `lhs` $\setminus$ `rhs` if both arguments are sets, the numerical difference otherwise.

`sl_div` implements the operator `/`.

* **Input:** `lhs` and `rhs` of type `Number`.
* **Output:** the *integer* quotient `lhs // rhs` if both arguments are integers, the floating point quotient otherwise.

The behaviour of `/` deserves an explanation.  The function `power` in `carmichael.sl` calls itself recursively
as `power(a, n / 2, m)` and stops when `n == 0`.  If `/` produced a floating point number, `n` would never become
the integer `0` and the recursion would not terminate.  For the same reason `{ 2 .. n / 2 }` in `primes.sl` needs
an integer bound.  Whenever a program really wants a floating point division, it writes at least one of the two
operands as a floating point number, exactly as `e.sl` does with `1.0 / factorial(i)`.

In [ ]:
def sl_add(lhs: Value, rhs: Value) -> Value:
    if isinstance(lhs, frozenset) and isinstance(rhs, frozenset):
        return lhs | rhs
    if isinstance(lhs, str) or isinstance(rhs, str):
        return to_string(lhs) + to_string(rhs)
    return lhs + rhs

def sl_sub(lhs: Value, rhs: Value) -> Value:
    if isinstance(lhs, frozenset) and isinstance(rhs, frozenset):
        return lhs - rhs
    return lhs - rhs

def sl_div(lhs: Number, rhs: Number) -> Number:
    if isinstance(lhs, int) and isinstance(rhs, int):
        return lhs // rhs
    return lhs / rhs

### Executing statements

The function `execute_tuple` loops through statements linearly.  The dictionary `Values` is the continuous state
that gets updated as assignments occur.

* **Input:** `StatementList`, a tuple of `NestedTuple`s, and `Values` of type `dict[str, Value]`.
* **Output:** `None`.  The statements are executed for their side effects on `Values`, on `Functions`, and on the screen.

In [ ]:
def execute_tuple(StatementList: tuple, Values: dict[str, Value]) -> None:
    for stmnt in StatementList:
        execute(stmnt, Values)

The function `execute` matches recursively against the shape of the nested tuple statement node, triggering
evaluation or recursive execution sequences.

* **Input:** `stmnt` of type `NestedTuple` and `Values` of type `dict[str, Value]`.
* **Output:** `None`.  A `return` statement does not return a value here but raises a `ReturnException`,
  which is caught in `call_function`.

Note that a function definition is executed like any other statement: executing it means *storing* the function
in the dictionary `Functions`.

In [ ]:
def execute(stmnt: NestedTuple, Values: dict[str, Value]) -> None:
    match stmnt:
        case ('program', *SL):
            execute_tuple(tuple(SL), Values)
        case ('function', ('IDENTIFIER', name), (',', *Parameters), body):
            Functions[name] = (tuple(parameter for _, parameter in Parameters), body)
        case (':=', ('IDENTIFIER', var), value):
            Values[var] = evaluate(value, Values)
        case ('expr', expr):
            evaluate(expr, Values)
        case ('return',):
            raise ReturnException(None)
        case ('return', expr):
            raise ReturnException(evaluate(expr, Values))
        case ('if', test, stmnt):
            if evaluate_bool(test, Values):
                execute(stmnt, Values)
        case ('if-else', test, stmnt, alternative):
            if evaluate_bool(test, Values):
                execute(stmnt, Values)
            else:
                execute(alternative, Values)
        case ('while', test, stmnt):
            while evaluate_bool(test, Values):
                execute(stmnt, Values)
        case ('for', init, test, update, stmnt):
            execute(init, Values)
            while evaluate_bool(test, Values):
                execute(stmnt, Values)
                execute(update, Values)
        case _:
            assert False, f'{stmnt} unexpected'

The function `evaluate_bool` evaluates the test of a conditional or of a loop and returns a pure boolean output.

* **Input:** `expr` of type `NestedTuple` and `Values` of type `dict[str, Value]`.
* **Output:** a `bool`.

In the original version of this notebook, `evaluate_bool` contained one case per comparison operator.  This is no
longer possible: a test may now be any expression, for instance the function call `is_prime(n)` or the
conjunction `gcd(a, n) == 1 && power(a, n-1, n) != 1`.  The comparisons have therefore moved into `evaluate`, and
`evaluate_bool` only converts the result into a truth value.  As in `C`, the number `0` and the empty set count
as *false*, everything else counts as *true*.

In [ ]:
def evaluate_bool(expr: NestedTuple, Values: dict[str, Value]) -> bool:
    return bool(evaluate(expr, Values))

### Calling a function

The function `call_function` performs a call of a user defined function.

* **Input:** `f_name` of type `str`, `Arguments`, a tuple of *unevaluated* `NestedTuple`s, and `Values` of type
  `dict[str, Value]`, the environment in which the arguments have to be evaluated.
* **Output:** the `Value` returned by the function, or `None` if the function ends without a `return` statement.

The arguments are evaluated in the environment of the *caller*, while the body is executed in a fresh dictionary
`Local` that binds only the parameters.  This is *call by value* with purely local variables.  If the name is not
the name of a user defined function, the call is passed on to `call_builtin`.

In [ ]:
def call_function(f_name: str, Arguments: tuple, Values: dict[str, Value]) -> Value | None:
    ArgumentValues = [evaluate(arg, Values) for arg in Arguments]
    if f_name in Functions:
        Parameters, body = Functions[f_name]
        assert len(Parameters) == len(ArgumentValues), \
               f'{f_name} expects {len(Parameters)} arguments, got {len(ArgumentValues)}'
        Local = dict(zip(Parameters, ArgumentValues))
        try:
            execute(body, Local)
        except ReturnException as result:
            return result.value
        return None
    return call_builtin(f_name, ArgumentValues)

The function `call_builtin` implements the predefined functions of our language.

* **Input:** `f_name` of type `str` and `Arguments`, a list of already evaluated `Value`s.
* **Output:** the `Value` computed by the built-in function.

| call            | meaning                                                                     |
|:----------------|:----------------------------------------------------------------------------|
| `print(x, ...)` | prints the arguments next to each other, followed by a newline, returns `0` |
| `read()`        | reads a natural number from the keyboard                                     |
| `len(S)`        | the number of elements of the set `S`                                        |
| `arb(S)`        | an arbitrary element of the non-empty set `S`                                |
| `cos(x)`, `sin(x)`, `exp(x)`, `sqrt(x)` | the corresponding functions of `math`                |
| `int(x)`, `float(x)` | conversion between the two numerical types                              |

`print` takes an arbitrary number of arguments and inserts *no* separator between them, so that
`print(i, "! = ", factorial(i))` produces a line like `5! = 120`.  In order to make the output of a program
reproducible, `arb` returns the *smallest* element with respect to `sort_key` instead of a genuinely arbitrary one.

In [ ]:
def call_builtin(f_name: str, Arguments: list) -> Value:
    match f_name:
        case 'print':
            print(''.join(to_string(arg) for arg in Arguments))
            return 0
        case 'read':
            return int(input('Please enter a natural number: '))
        case 'len':
            return len(Arguments[0])
        case 'arb':
            S = Arguments[0]
            assert len(S) > 0, 'arb applied to the empty set'
            return min(S, key=sort_key)
        case 'cos':
            return math.cos(Arguments[0])
        case 'sin':
            return math.sin(Arguments[0])
        case 'exp':
            return math.exp(Arguments[0])
        case 'sqrt':
            return math.sqrt(Arguments[0])
        case 'int':
            return int(Arguments[0])
        case 'float':
            return float(Arguments[0])
        case _:
            assert False, f'unknown function {f_name}'

### Evaluating expressions

The function `evaluate` calculates the value of an expression.  We utilize python's built-in `match`/`case` to
guarantee exhaustive mapping of our AST types.

* **Input:** `expr` of type `NestedTuple` and `Values` of type `dict[str, Value]`.
* **Output:** the `Value` of the expression.

The leaves of the abstract syntax tree are tokens:  A token of type `NUMBER` or `FLOAT` is converted into a number,
while the text of a token of type `STRING` is stripped of its double quotes and unescaped.  A token of type
`IDENTIFIER` is a variable, whose value is looked up in `Values`.

Furthermore, the operators `&&` and `||` are evaluated *lazily*: in the test `gcd(a, n) == 1 && power(a, n-1, n) != 1`
the expensive call of `power` is only performed if `a` and `n` are coprime.

In [ ]:
def evaluate(expr: NestedTuple, Values: dict[str, Value]) -> Value:
    match expr:
        case ('NUMBER', digits):
            return int(digits)
        case ('FLOAT', digits):
            return float(digits)
        case ('STRING', text):
            return unescape(text[1:-1])
        case ('IDENTIFIER', name):
            assert name in Values, f'undefined variable {name}'
            return Values[name]
        case ('call', ('IDENTIFIER', f_name), *Arguments):
            return call_function(f_name, tuple(Arguments), Values)
        case ('set', *Elements):
            return frozenset(evaluate(element, Values) for element in Elements)
        case ('..', lower, upper):
            return frozenset(range(evaluate(lower, Values), evaluate(upper, Values) + 1))
        case ('comprehension', body, (',', *Generators)):
            return evaluate_comprehension(body, tuple(Generators), None, Values)
        case ('comprehension', body, (',', *Generators), condition):
            return evaluate_comprehension(body, tuple(Generators), condition, Values)
        case ('+', lhs, rhs):
            return sl_add(evaluate(lhs, Values), evaluate(rhs, Values))
        case ('-', lhs, rhs):
            return sl_sub(evaluate(lhs, Values), evaluate(rhs, Values))
        case ('*', lhs, rhs):
            return evaluate(lhs, Values) *  evaluate(rhs, Values)
        case ('/', lhs, rhs):
            return sl_div(evaluate(lhs, Values), evaluate(rhs, Values))
        case ('%', lhs, rhs):
            return evaluate(lhs, Values) %  evaluate(rhs, Values)
        case ('neg', arg):
            return -evaluate(arg, Values)
        case ('==', lhs, rhs):
            return evaluate(lhs, Values) == evaluate(rhs, Values)
        case ('!=', lhs, rhs):
            return evaluate(lhs, Values) != evaluate(rhs, Values)
        case ('<=', lhs, rhs):
            return evaluate(lhs, Values) <= evaluate(rhs, Values)
        case ('>=', lhs, rhs):
            return evaluate(lhs, Values) >= evaluate(rhs, Values)
        case ('<', lhs, rhs):
            return evaluate(lhs, Values) <  evaluate(rhs, Values)
        case ('>', lhs, rhs):
            return evaluate(lhs, Values) >  evaluate(rhs, Values)
        case ('&&', lhs, rhs):
            return evaluate_bool(lhs, Values) and evaluate_bool(rhs, Values)
        case ('||', lhs, rhs):
            return evaluate_bool(lhs, Values) or  evaluate_bool(rhs, Values)
        case ('!', arg):
            return not evaluate_bool(arg, Values)
        case _:
            assert False, f'{expr} unexpected'

The function `evaluate_comprehension` computes the value of a set comprehension of the form
$\{ \texttt{body} : x \in S,\; y \in T \mid \texttt{condition} \}$.

* **Input:** `body` of type `NestedTuple`, `Generators`, a tuple of triples of the form `('in', ('IDENTIFIER', variable), set_expr)`,
  `condition` of type `NestedTuple` or `None`, and `Values` of type `dict[str, Value]`.
* **Output:** a `frozenset` containing the value of `body` for every combination of bindings that satisfies the condition.

The generators are worked off from left to right by the local function `loop`, which binds one variable per level
of recursion.  Later generators may depend on earlier ones, which is what `{ p * q : p in {2 .. n/2}, q in {p .. n/p} }`
in `primes.sl` relies upon.  The bound variables are written into `Values` itself, so after the comprehension has
been evaluated they are still visible; this mirrors the behaviour of a `for` loop in our language.

In [ ]:
def evaluate_comprehension(body:       NestedTuple,
                          Generators: tuple,
                          condition:  NestedTuple | None,
                          Values:     dict[str, Value]) -> frozenset:
    Result = []
    def loop(index: int) -> None:
        if index == len(Generators):
            if condition is None or evaluate_bool(condition, Values):
                Result.append(evaluate(body, Values))
            return
        _, (_, variable), source = Generators[index]
        for element in sorted(evaluate(source, Values), key=sort_key):
            Values[variable] = element
            loop(index + 1)
    loop(0)
    return frozenset(Result)

## Putting Everything Together

Finally, we bring the whole pipeline together.  `main` will open the program file, parse it into a nested tuple,
and evaluate it under an isolated state dict.

* **Input:** `file_name` of type `str`, and the optional flag `show_ast` of type `bool`.
* **Output:** `None`.  The program is executed for its side effects.

The syntax tree is only drawn if `show_ast` is set to `True`, since the trees of programs like `carmichael.sl`
are far too big to be readable.  The dictionary `Functions` is cleared at the start of every run so that
functions defined by one program cannot leak into the next.

In [ ]:
def main(file_name: str, show_ast: bool=False) -> None:
    with open(file_name, 'r') as handle:
        program = handle.read()
    ast = parser.parse(program)
    if show_ast:
        display(tuple2dot(ast))
    Functions.clear()
    Values: dict[str, Value] = {}
    execute(ast, Values)

## Running the Example Programs

### `factorial.sl`

A recursive function together with a `for` loop.  Since `Python` integers have arbitrary precision, `25!` is
computed exactly.

In [ ]:
!cat factorial.sl

In [ ]:
main('factorial.sl', show_ast=True)

### `sum-for.sl`

This program calls the built-in function `read`, so executing the following cell will ask you for a number.
Enter for example `10`; the program then computes $1 + 2 + \dots + 100 = 5050$.

In [ ]:
!cat sum-for.sl

In [ ]:
main('sum-for.sl')

### `e.sl`

Euler's number as the sum $\sum\limits_{i=0}^{20} \frac{1}{i!}$.  Here `1.0 / factorial(i)` mixes a floating
point number with an integer and therefore yields a floating point number.

In [ ]:
!cat e.sl

In [ ]:
main('e.sl')

### `pi.sl`

The *Bailey-Borwein-Plouffe* series for $\pi$.  The program prints the current approximation in every step, so
you can watch it converge after about a dozen iterations.

In [ ]:
!cat pi.sl

In [ ]:
main('pi.sl')

### `solve.sl`

The fixed point of the cosine, computed both iteratively and recursively.  This program uses the built-in
function `cos` and the floating point literal `1.0e-15`.

In [ ]:
!cat solve.sl

In [ ]:
main('solve.sl')

### `power.sl`

The power set of $\{1, \dots, 4\}$.  This is the program that forces us to represent sets as `frozenset`s: the
result is a *set of sets*.

In [ ]:
!cat power.sl

In [ ]:
main('power.sl')

### `primes.sl`

The sieve of Eratosthenes written as a single set comprehension with two generators: a number is prime iff it is
not a product $p \cdot q$ with $p, q \geq 2$.

In [ ]:
!cat primes.sl

In [ ]:
main('primes.sl')

### `carmichael.sl`

The *Carmichael numbers* below $10\,000$, that is the composite numbers $n$ that satisfy Fermat's little theorem
$a^{n-1} \equiv 1 \;(\textrm{mod}\; n)$ for every $a$ that is coprime to $n$.  This is by far the most demanding
of our programs, and it exercises almost every feature of the language at once: recursion, `for` loops, `&&`,
`||`, and set comprehensions.  Expect a runtime of roughly ten seconds.

In [ ]:
!cat carmichael.sl

In [ ]:
main('carmichael.sl')

## Exercises

1. Our language has no `else if` chain and no `break` statement.  Extend the grammar and the function `execute`
   accordingly.  For `break` you can reuse the trick that is already used for `return`.
2. Currently a function cannot read a global variable at all.  Change `call_function` so that a name that is not
   a parameter is looked up in the global dictionary.  Which of the example programs change their behaviour?
3. The function `is_prime` in `carmichael.sl` uses the test `t * t < n` and therefore claims that `9` is prime.
   Why does `is_carmichael` nevertheless produce the correct answer?
4. Add a built-in function `pow(a, n, m)` that computes $a^n \;\textrm{mod}\; m$ and compare the runtime of
   `carmichael.sl` when the interpreted function `power` is replaced by it.  What does the difference tell you
   about the cost of interpretation?